# 04. Validation Strategy and Cross Validation

validation 결과 한 번만 보고 모델을 확정해도 괜찮을까?

**목표**
1. hold-out validation의 장점과 한계를 이해한다.
2. cross validation의 목적을 이해한다.
3. 평균 점수뿐 아니라 fold별 점수의 흔들림도 함께 읽는다.
4. 높은 점수의 모델과 안정적인 모델을 함께 비교하는 시각을 만든다.

## 1. validation 결과 한 번만 보고 결정하면 왜 위험할까

validation split 하나의 결과는 우연의 영향을 받을 수 있다.
어떤 split은 비교적 쉬울 수 있고, 어떤 split은 더 어려울 수 있다.

즉, validation 결과 한 번만 보고 이 모델이 항상 더 좋다고 단정하면 위험할 수 있다.

## 2. hold-out validation의 장점과 한계

**장점**
- 구조가 단순하다.
- train / validation / test의 역할을 이해하기 쉽다.

**한계**
- split 하나에 결과가 의존할 수 있다.
- validation set이 우연히 쉬우면 모델을 과신할 수 있다.
- validation set이 우연히 어려우면 좋은 모델을 과소평가할 수도 있다.

즉, hold-out validation은 매우 중요하지만,
필요한 상황에서는 cross validation으로 한 번 더 안정성을 확인할 수 있다.

## 3. cross validation의 목적

cross validation은 검증을 더 안정적으로 하기 위한 방법이다.

1. 데이터를 여러 번 나누어 반복 검증한다.
2. 한 번의 validation 결과에만 의존하지 않는다.
3. 평균 성능과 성능의 흔들림을 함께 본다.

즉, 이번 split에서 우연히 잘된 것인가 아니면 대체로 안정적으로 잘되는가
를 조금 더 잘 판단하기 위한 도구라고 볼 수 있다.

## 4. K-Fold

- hold-out validation: 1번 나누어 검증
- K-Fold cross validation: 여러 번 나누어 반복 검증

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.datasets import make_classification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 5. 회귀 데이터 준비

In [ ]:
rng = np.random.default_rng(SEED)
n_samples = 4000

X = pd.DataFrame({
    'feature_00': rng.normal(0, 1.0, n_samples),
    'feature_01': rng.normal(0, 1.2, n_samples),
    'feature_02': rng.uniform(-2.0, 2.0, n_samples),
    'feature_03': rng.uniform(-3.0, 3.0, n_samples),
    'feature_04': rng.normal(0, 1.0, n_samples),
    'feature_05': rng.normal(0, 1.0, n_samples),
    'feature_06': rng.normal(0, 1.0, n_samples),
    'feature_07': rng.normal(0, 1.0, n_samples),
    'feature_08': rng.exponential(1.0, n_samples),
    'feature_09': rng.normal(0, 1.0, n_samples),
    'feature_10': rng.normal(0, 1.0, n_samples),
    'feature_11': rng.normal(0, 1.0, n_samples),
})

base_noise = rng.normal(0, 1.1 + 0.9 * np.abs(X['feature_07']), n_samples)
outlier_mask = rng.random(n_samples) < 0.03
base_noise[outlier_mask] += rng.normal(0, 7.5, outlier_mask.sum())

y = (
    5.0 * X['feature_00']
    - 3.2 * X['feature_01']
    + 2.4 * (X['feature_02'] ** 2)
    + 1.9 * np.sin(1.4 * X['feature_03'])
    + 2.7 * (X['feature_04'] * X['feature_05'])
    + 1.5 * np.maximum(X['feature_06'], 0)
    + 0.9 * np.log1p(X['feature_08'])
    + base_noise
)

print('X shape:', X.shape)
print('y shape:', y.shape)

## 6. 회귀 모델 후보 준비

- Ridge: 선형 baseline
- RandomForestRegressor: 비선형성과 interaction을 더 잘 잡을 수 있는 후보

이번에는 validation 한 번의 점수가 아니라, cross validation 기준으로 결과가 어떻게 보이는지 확인한다.

In [ ]:
ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=3.0))
])

rf_model = RandomForestRegressor(
    n_estimators=220,
    max_depth=12,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1
)

## 7. hold-out 결과와 cross validation 결과 비교

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    random_state=SEED
)

ridge_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

ridge_holdout_rmse = np.sqrt(np.mean((y_val - ridge_model.predict(X_val)) ** 2))
rf_holdout_rmse = np.sqrt(np.mean((y_val - rf_model.predict(X_val)) ** 2))

print('ridge hold-out rmse:', round(ridge_holdout_rmse, 4))
print('rf hold-out rmse   :', round(rf_holdout_rmse, 4))

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

ridge_cv_scores = cross_val_score(
    ridge_model,
    X, y,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

rf_cv_scores = cross_val_score(
    rf_model,
    X, y,
    cv=cv,
    scoring='neg_root_mean_squared_error'
)

ridge_rmse_scores = -ridge_cv_scores
rf_rmse_scores = -rf_cv_scores

print('ridge fold rmse:', np.round(ridge_rmse_scores, 4))
print('rf fold rmse   :', np.round(rf_rmse_scores, 4))

In [ ]:
# TODO: hold-out rmse, cv rmse 평균, cv rmse 표준편차, cv rmse 범위, cv 표준편차 비율을 한 테이블로 보기 좋게 정리해서 출력
reg_cv_result_df = pd.DataFrame({
    'model': ['ridge_baseline', 'random_forest_improved'],
    'holdout_rmse': [ridge_holdout_rmse, rf_holdout_rmse],
    'cv_rmse_mean': [____, ____],
    'cv_rmse_std': [____, ____],
    'cv_rmse_range': [____, ____],
    'cv_std_ratio_pct': [____, ____]
})

reg_cv_result_df.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, 6), ridge_rmse_scores, marker='o', label='ridge_baseline')
plt.plot(range(1, 6), rf_rmse_scores, marker='o', label='random_forest_improved')
plt.title('Fold-wise RMSE Comparison')
plt.xlabel('fold')
plt.ylabel('rmse')
plt.legend()
plt.show()

## 8. 회귀 cross validation 결과를 어떻게 읽을까

1. 평균 RMSE가 더 낮은가
   - 전반적인 성능이 더 좋다고 볼 수 있다.

2. 표준편차가 큰가 작은가
   - split이 바뀌었을 때 성능이 얼마나 흔들리는지 볼 수 있다.

3. hold-out 결과와 CV 평균이 비슷한가
   - hold-out 결과가 지나치게 낙관적이거나 비관적이지 않았는지 참고할 수 있다.

cross validation은 hold-out을 무효화하는 것이 아니라,
hold-out 판단을 조금 더 안정적으로 보정해주는 역할을 한다고 이해하면 좋다.

## 9. 분류에서는 왜 StratifiedKFold일까

분류에서는 클래스 비율이 fold마다 크게 달라지면 평가가 왜곡될 수 있다.

그래서 분류에서는 보통 StratifiedKFold를 많이 사용한다.
즉,
- 회귀: KFold를 자주 사용
- 분류: StratifiedKFold를 자주 사용

로 정리하면 된다.

## 10. 분류 예시: fold별 F1 점수 다시 보기

In [ ]:
X_cls, y_cls = make_classification(
    n_samples=2500,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.75, 0.25],
    flip_y=0.03,
    class_sep=0.9,
    random_state=SEED
)

logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

hgb_model = HistGradientBoostingClassifier(random_state=SEED)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
logistic_f1_scores = cross_val_score(
    logistic_model,
    X_cls, y_cls,
    cv=skf,
    scoring='f1'
)

hgb_f1_scores = cross_val_score(
    hgb_model,
    X_cls, y_cls,
    cv=skf,
    scoring='f1'
)

print('logistic fold f1:', np.round(logistic_f1_scores, 4))
print('hgb fold f1     :', np.round(hgb_f1_scores, 4))

In [ ]:
# TODO: f1_mean, f1_std, f1_range, f1_std_ratio_pct을 한 테이블로 보기 좋게 정리해서 출력
cls_cv_result_df = pd.DataFrame({
    'model': ['logistic_regression', 'hist_gradient_boosting'],
    'f1_mean': [____, ____],
    'f1_std': [____, ____],
    'f1_range': [____, ____],
    'f1_std_ratio_pct': [____, ____]
})

cls_cv_result_df.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, 6), logistic_f1_scores, marker='o', label='logistic_regression')
plt.plot(range(1, 6), hgb_f1_scores, marker='o', label='hist_gradient_boosting')
plt.title('Fold-wise F1 Comparison')
plt.xlabel('fold')
plt.ylabel('f1')
plt.legend()
plt.show()

## 11. 평균 점수와 안정성은 함께 봐야 한다

어떤 모델은 평균 점수는 높지만 fold별 편차가 클 수 있다.
반대로 어떤 모델은 평균 점수는 조금 낮아도 더 안정적일 수 있다.

이 둘을 함께 봐야 한다.

예를 들어
- 점수 차이가 아주 크다면 높은 점수 쪽이 자연스럽다.
- 점수 차이가 작다면 더 단순하거나 안정적인 모델도 좋은 선택이 될 수 있다.

즉, best score와 best choice가 항상 같지는 않다.

## 정리

1. hold-out validation은 매우 중요한 출발점이다.
2. 하지만 validation 결과 한 번만으로 모델을 확정하면 위험할 수 있다.
3. cross validation은 평균 성능과 점수의 흔들림을 함께 보게 해준다.
4. 좋은 모델은 높은 점수뿐 아니라 안정성도 함께 고려해야 한다.